In [1]:
from dotenv import load_dotenv
load_dotenv()
import os
from supabase import create_client, Client
supabase_url = os.environ.get("SUPABASE_URL")
supabase_api_key = os.environ.get("SUPBASE_KEY")

supabase: Client = create_client(supabase_url, supabase_api_key)

In [2]:
from openai import OpenAI
client = OpenAI(api_key = os.getenv("OPENAI_API_KEY"))

In [3]:
def report_research(date, next_date):
    result = supabase.table('curation_selected_items')\
        .select('event_id, output,sources, news_date','topic, created_at')\
    .gte('created_at', date)\
    .lt('created_at', next_date)\
    .eq('topic', 'Report')\
        .order('created_at', desc=False)\
        .execute()
    
    return result.data

In [4]:
report_deep_dive = report_research('2026-03-04', '2026-03-05')

In [5]:
report_deep_dive

[{'event_id': '1945_2026-03-02',
  'output': "Palo Alto Networks Unit42 published a detailed advisory (March 2, 2026) on CVE-2026-0628: a high-severity vulnerability in Chrome's new Gemini Live panel that allowed malicious extensions with basic permissions to hijack the Gemini panel, potentially enabling access to local files, camera/microphone, screenshots, and privilege escalation. Unit42 states they responsibly disclosed the issue (disclosed Oct 23, 2025) and that Google issued a fix in early January 2026; the advisory emphasizes enterprise risk from AI-enabled browser features and recommends mitigations and incident response contacts.",
  'sources': [{'url': 'https://unit42.paloaltonetworks.com/gemini-live-in-chrome-hijacking/',
    'name': 'Palo Alto Networks Unit42'}],
  'news_date': '2026-03-02',
  'topic': 'Report',
  'created_at': '2026-03-04T05:58:54.375098+00:00'},
 {'event_id': '1942_2026-03-02',
  'output': "Reuters reports the U.S. Federal Reserve is accelerating work to 

In [9]:
b = [{'event_id': '2074_2026-03-03',
  'output': 'Nutanix published its Enterprise Cloud Index (March 3, 2026) reporting that AI is accelerating container adoption and driving infrastructure modernization: 85% of respondents say AI is accelerating container adoption; 87% expect container use for applications to increase over three years; 82% say silos create AI risks; 79% report shadow IT AI usage; 61% expect AI agents to enhance experiences. Findings are based on a global survey of 1,600 cloud/IT executives and quantify adoption trends and operational risks for enterprise infrastructure.',
  'sources': [{'url': 'https://ir.nutanix.com/news-releases/news-release-details/nutanix-enterprise-cloud-index-finds-ai-driving-rapid-container',
    'name': 'Nutanix'}],
  'news_date': '2026-03-03',
  'topic': 'Report',
  'created_at': '2026-03-04T06:01:19.166042+00:00'}]

In [6]:
def openai_research_workflow(research_input):
    for i in research_input:
        response = client.responses.create(
            model = "gpt-5-nano",
            tools = [{
                "type": "web_search"
            }],
            include = ["web_search_call.action.sources"],
            input = f"""
            You are a senior research analyst for Krux.

            TASK:
              Create a deep, fact-checked brief for ONE AI report/paper/benchmark release.
            
            GOAL:
              Extract the highest-value findings and implementation guidance so a later 100-word summary can capture most practical signal.
            
            OUTPUT FORMAT:
              - 12 to 16 bullet points.
              - Each bullet must include inline citations:
                [Source: <publisher>, <url>]
              - No text before or after bullets.
            
            MANDATORY STRUCTURE:
              1) REPORT SNAPSHOT
              - What was published, by whom, and when.
              - Scope: geography, sectors, sample size, time window, method type.
            
              2) CORE FINDINGS (MOST IMPORTANT)
              - 4 to 6 bullets with the strongest quantified findings.
              - Include exact numbers, not vague language.
            
              3) WHAT THIS ACTUALLY MEANS
              - 2 to 3 bullets translating findings into plain English implications for real teams.
            
              4) IMPLEMENTATION PLAYBOOK
              - 3 to 4 bullets: what organizations should do in the next quarter.
              - Include sequencing (e.g., data readiness -> pilot -> measurement -> rollout).
            
              5) METRICS TO TRACK
              - 1 or 2 bullets on KPIs teams should monitor to apply the report in practice.
            
              6) LIMITATIONS / CAVEATS
              - 1 or 2 bullets on methodology limits, sample bias, missing data, or uncertainty.
            
              7) DECISION TAKEAWAY
              - 1 bullet: “If a team only does one thing based on this report, it should be X.”
            
              STRICT RULES:
              - No hype, no opinion, no generic AI commentary.
              - Every factual claim must be source-backed.
              - If a key detail is missing, write: "Not disclosed in cited sources."
              - Prefer primary sources (original report/paper) over secondary articles.
              - If secondary coverage conflicts with primary report, prioritize primary and note conflict.
            
              QUALITY CHECK:
              - Must contain quantified findings.
              - Must contain actionable implementation steps.
              - Must clearly separate findings from interpretation.
              - Must include limitations.
            
              Report/event to research:
              Report summary: {i['output']}
              Source: {i['sources']}
            """,
        )

        output = response.output_text
        print(output)

        final_dictionary = {
            'event_id': i['event_id'],
            'news_date': i['news_date'],
            'output': output,
            'model_provider': 'openai',
            'topic': i['topic']
        }
        print(final_dictionary)

        save_research(final_dictionary)

        print("✅ Done")

In [10]:
openai_research_workflow(b)

- REPORT SNAPSHOT: Nutanix published the eighth annual Enterprise Cloud Index (ECI) survey findings on March 3, 2026, authored by Nutanix (NASDAQ: NTNX). [Source: Nutanix, https://www.nutanix.com/press-releases/2026/nutanix-enterprise-cloud-index-finds-ai-is-driving-rapid-container-adoption]

- REPORT SNAPSHOT: Scope and methodology—global survey of 1,600 cloud, IT, and engineering executives with at least manager-level titles from organizations with 500+ employees, across 14 markets, conducted in November 2025 by Wakefield Research. [Source: Nutanix, https://www.nutanix.com/press-releases/2026/nutanix-enterprise-cloud-index-finds-ai-is-driving-rapid-container-adoption]

- CORE FINDINGS: 85% of respondents report that AI is accelerating their adoption of containers. [Source: Nutanix, https://www.nutanix.com/press-releases/2026/nutanix-enterprise-cloud-index-finds-ai-is-driving-rapid-container-adoption]

- CORE FINDINGS: 87% of respondents expect container use for applications to increa

In [7]:
def save_research(research_json):
    supabase.table('research_assistant').insert({
        'event_id': research_json['event_id'],
        'model_provider': research_json['model_provider'],
        'news_date': research_json['news_date'],
        'output': research_json['output'],
        'topic': research_json['topic']
    }).execute()